In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.ticker import ScalarFormatter, NullFormatter
import numpy as np

# Import CSV file
df = pd.read_csv('Date_110226.csv', names=['id', 'snp', 'date', 'group'], skiprows=1)

# Create a mapping for colors and markers based on 'group'
group_colors = {
    'KNC': u'#8c564b',
    'BCI': u'#bcbd22',
    'DDL': u'#ff7f0e',
    'GCE': u'#f72585',
    'MLI': u'#ffd60a',
    'PGN': u'#003049',
    'PLO': u'#1f77b4',
    'SND': u'#9e0059',
    'VRD': u'#17becf',
    'VRV': u'#219ebc'
}
group_markers = {
    'KNC': 'o',
    'BCI': 'o',
    'DDL': 'o',
    'GCE': 'o',
    'MLI': 'o',
    'PGN': 'o',
    'PLO': 'o',
    'SND': 'o',
    'VRD': 'o',
    'VRV': 'o'
}

# Map colors and markers
df['color'] = df['group'].map(group_colors)
df['marker'] = df['group'].map(group_markers)

# Add jitter to x and y coordinates
np.random.seed(42)  # For reproducibility
jitter_strength_x = 40  # Adjust jitter strength for x-coordinates
jitter_strength_y = 0.05  # Adjust jitter strength for y-coordinates
df['date_jittered'] = df['date'] + np.random.uniform(-jitter_strength_x, jitter_strength_x, size=len(df))
#df['snp_jittered'] = df['snp'] * (1 + np.random.uniform(-jitter_strength_y, jitter_strength_y, size=len(df)))

# Create a figure and axis
fig = plt.figure(figsize=(12, 3), dpi=600)
fig.patch.set_facecolor('white')  # Set canvas background to white
ax = fig.add_subplot(111)

# Add shaded background regions
distinct_colors = ['#d0e0dd', '#e4d7e7', '#f6d6bd', '#d6dbee']  # Custom colors for periods
bounds = [200, 500, 750, 1050, 1700]  # Boundaries for shading
for i in range(len(bounds) - 1):
    ax.axvspan(bounds[i], bounds[i + 1], color=distinct_colors[i], alpha=0.3)

# Add frame within specific x and y bounds (1500 to 500 BCE and specific y)
ax.fill_between([502, 1698], 530000, 1320000, color='black', alpha=0.1)


# First, plot KNC to be underneath
if 'KNC' in df['group'].unique():
    knc_df = df[df['group'] == 'KNC']
    ax.scatter(knc_df['date_jittered'], knc_df['snp'], s=40, c=group_colors['KNC'],
               marker=group_markers['KNC'], label='KNC', edgecolor='k', alpha=0.8, zorder=2)

# Then plot the rest (excluding KNC)
for group, group_df in df.groupby('group'):
    if group == 'KNC':
        continue
    ax.scatter(group_df['date_jittered'], group_df['snp'], s=40, c=group_colors[group],
               marker=group_markers[group], label=group, edgecolor='k', alpha=0.8, zorder=3)

# Set axis scaling and limits
ax.set_yscale('log')
ax.set_yticks([100000, 1000000])
for axis in [ax.yaxis]:
    axis.set_major_formatter(ScalarFormatter())
    axis.set_minor_formatter(NullFormatter())
ax.set_xlim([500, 1700])  # x-axis range remains as per original code
ax.set_xticks([200, 500, 750, 1050, 1700])
ax.set_xticklabels([200, 500, 750, 1050, 1700], fontsize=16)  # Label x-ticks with numbers
ax.set_ylim([0, 1500000])
ax.set_ylabel('# SNPs covered', rotation=90, fontsize=18)
ax.tick_params(labelsize=16)
ax.invert_xaxis()
ax.set_xlabel('Date (BCE)', fontsize=18, labelpad=10)

# Add period names between x ticks (adjust y-position as needed)
period_labels = ['Hellenistic', 'DIA', 'EIA', 'MLBA']
label_positions = [0.82, 0.68, 0.54, 0.3]  # Approximate horizontal locations (adjust if needed)
y_fig = 0.96  # Vertical position in figure coordinates (just below the plot)

for label, x_fig in zip(period_labels, label_positions):
    fig.text(x_fig, y_fig, label, ha='center', va='top', fontsize=16)

ax.yaxis.get_offset_text().set_fontsize(16)

# Custom legend with KNC at the top
handles, labels = ax.get_legend_handles_labels()
legend_order = ['KNC'] + sorted([lbl for lbl in labels if lbl != 'KNC'])
ordered_handles = [handles[labels.index(l)] for l in legend_order]
#ax.legend(ordered_handles, legend_order, fontsize=10, loc=(1.05, 0))


# Show the plot
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.ticker import ScalarFormatter, NullFormatter
import numpy as np
import matplotlib.lines as mlines

# Import main data
df = pd.read_csv('Date_110226.csv', names=['id', 'snp', 'date', 'group'], skiprows=1)

# Import IBD connections
ibd_df = pd.read_csv(
    '/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_16cM_fracgp0.7.csv',
    #names=['IID1', 'IID2', 'lengthIBD'],
    skiprows=1
)
ibd_ids = pd.unique(ibd_df[['iid1', 'iid2']].values.ravel())

# Load frac_gp info
frac_df = pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/iid.KNC.v12.0.tsv', sep='\t')
frac_df=frac_df[frac_df['frac_gp']>=0.7]
# convert to sets for clean comparison
df_ids = set(df['id'])
frac_ids = set(frac_df['iid'])   # adjust column name if different

# IDs in frac_df but NOT in df
missing_ids = frac_ids - df_ids

print(f"Total IDs in frac_df: {len(frac_ids)}")
print(f"Total IDs in df: {len(df_ids)}")
print(f"Missing IDs (in frac_df but not in df): {len(missing_ids)}")

if missing_ids:
    print("Example missing IDs:")
    print(list(missing_ids)[:10])
else:
    print("All frac_df IIDs are present in df.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.ticker import ScalarFormatter, NullFormatter
import numpy as np
import matplotlib.lines as mlines

# Import main data
df = pd.read_csv('Date_110226.csv', names=['id', 'snp', 'date', 'group'], skiprows=1)

# Import IBD connections
ibd_df = pd.read_csv(
    '/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_16cM_fracgp0.7.csv'
    #names=['IID1', 'IID2', 'lengthIBD'],
    #skiprows=1
)
ibd_ids = pd.unique(ibd_df[['iid1', 'iid2']].values.ravel())

# Load frac_gp info
frac_df = pd.read_csv('/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/iid.KNC.v12.0.tsv', sep='\t')

# Merge frac_gp into df
df = df.merge(frac_df[['iid', 'frac_gp']], how='left', left_on='id', right_on='iid')

# Filter to only individuals in the IBD file
df = df[df['id'].isin(ibd_ids)]

# Color and marker mappings
group_colors = {
    'KNC': u'#8c564b', 'BCI': u'#bcbd22', 'DDL': u'#ff7f0e', 'GCE': u'#f72585',
    'MLI': u'#ffd60a', 'PGN': u'#003049', 'PLO': u'#1f77b4', 'SND': u'#9e0059',
    'VRD': u'#17becf', 'VRV': u'#219ebc'
}
group_markers = {key: 'o' for key in group_colors}
group_markers['SND'] = 'd'

df['color'] = df['group'].map(group_colors)
df['marker'] = df['group'].map(group_markers)

# Jitter date
np.random.seed(42)
jitter_strength_x = 40
df['date_jittered'] = df['date'] + np.random.uniform(-jitter_strength_x, jitter_strength_x, size=len(df))

# Figure setup
fig = plt.figure(figsize=(12, 7), dpi=600)
fig.patch.set_facecolor('white')
ax = fig.add_subplot(111)

# Shaded regions
period_colors = ['#d0e0dd', '#e4d7e7', '#f6d6bd', '#d6dbee']
bounds = [200, 500, 750, 1050, 1700]
for i in range(len(bounds) - 1):
    ax.axvspan(bounds[i], bounds[i + 1], color=period_colors[i], alpha=0.3)

# Scatter plot for grouped individuals
for group, group_df in df.groupby('group'):
    ax.scatter(
        group_df['date_jittered'], group_df['frac_gp'],
        s=90, c=group_colors[group], marker=group_markers[group],
        label=group, edgecolor='k', zorder=2
    )

# Plot IBD lines
for i, row in ibd_df.iterrows():
    iid1, iid2 = row['iid1'], row['iid2']
    p1 = df[df['id'] == iid1]
    p2 = df[df['id'] == iid2]
    if not p1.empty and not p2.empty:
        ax.plot(
            [p1['date_jittered'].iloc[0], p2['date_jittered'].iloc[0]],
            [p1['frac_gp'].iloc[0], p2['frac_gp'].iloc[0]],
            color='#74a892', linewidth=0.5, alpha=0.5, zorder=1
        )

# Now handle extra KNCs with frac_gp > 0.7 not in IBD
knc_extra = frac_df[(frac_df['frac_gp'] > 0.7) & (frac_df['iid'].str.startswith('KNC'))]
full_df = pd.read_csv('Date_150425.csv', names=['id', 'snp', 'date', 'group'], skiprows=1)
extra_df = full_df[full_df['id'].isin(knc_extra['iid'])]
extra_df = extra_df[~extra_df['id'].isin(ibd_ids)]
extra_df = extra_df.merge(frac_df[['iid', 'frac_gp']], how='left', left_on='id', right_on='iid')
extra_df['date_jittered'] = extra_df['date'] + np.random.uniform(-jitter_strength_x, jitter_strength_x, size=len(extra_df))

# Plot extra KNCs
ax.scatter(
    extra_df['date_jittered'], extra_df['frac_gp'],
    s=90, c=group_colors['KNC'], marker=group_markers['KNC'],
    edgecolor='k', zorder=2
)

# Return extra non-IBD KNCs
print("KNC individuals with frac_gp > 0.7 not in IBD file:", extra_df['id'].tolist())

# Final axis setup
ax.set_ylim(0.685, 0.99)
ax.set_ylabel('Imputed genotype likelihood score', fontsize=18)
ax.set_yticks([0.7, 0.8, 0.9])
ax.set_yticklabels([0.7, 0.8, 0.9], fontsize=16)
ax.set_xlim([500, 1700])
ax.set_xticks([500, 750, 1050, 1700])
ax.set_xticklabels([500, 750, 1050, 1700], fontsize=16)
ax.tick_params(labelsize=16)
ax.invert_xaxis()
ax.set_xlabel('Date (BCE)', fontsize=18, labelpad=10)

# Show plot
plt.show()
